# Notebook 1: Data Loading & Preparation
**NYC Taxi Fare Prediction — Nesha's Work**

This notebook contains the process of loading and preparing data using PySpark.

**Dataset:** NYC Taxi Fare Prediction  
**Columns:** key, fare_amount, pickup_datetime, pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude, passenger_count

In [16]:
import sys
print(sys.executable)


c:\Program Files\Python311\python.exe


## 1.1 Setup & Initialize PySpark

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType
)

print('PySpark successfully imported')
print(f'Python version: {sys.version}')

PySpark berhasil diimport
Python version: 3.11.6 (tags/v3.11.6:8b6ee5b, Oct  2 2023, 14:57:12) [MSC v.1935 64 bit (AMD64)]


In [ ]:
# Create SparkSession
spark = SparkSession.builder \
    .appName('NYC_Taxi_Fare_Prediction') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

# Set log level to not be too verbose
spark.sparkContext.setLogLevel('WARN')

print(f'Spark Version: {spark.version}')
print(f'SparkSession successfully created: {spark.sparkContext.appName}')

Spark Version: 4.1.1
SparkSession berhasil dibuat: NYC_Taxi_Fare_Prediction


## 1.2 Define Data Schema

Defining schema explicitly is more efficient than letting PySpark infer automatically, especially for large datasets.

In [ ]:
# Explicitly define schema
schema = StructType([
    StructField('key',                StringType(),  True),
    StructField('fare_amount',        FloatType(),   True),
    StructField('pickup_datetime',    StringType(),  True),
    StructField('pickup_longitude',   FloatType(),   True),
    StructField('pickup_latitude',    FloatType(),   True),
    StructField('dropoff_longitude',  FloatType(),   True),
    StructField('dropoff_latitude',   FloatType(),   True),
    StructField('passenger_count',    IntegerType(), True),
])

print('Schema successfully defined:')
for field in schema.fields:
    print(f'  - {field.name}: {field.dataType}')

Schema berhasil didefinisikan:
  - key: StringType()
  - fare_amount: FloatType()
  - pickup_datetime: StringType()
  - pickup_longitude: FloatType()
  - pickup_latitude: FloatType()
  - dropoff_longitude: FloatType()
  - dropoff_latitude: FloatType()
  - passenger_count: IntegerType()


## 1.3 Load Dataset

In [ ]:
# Path to CSV file (adjust if different)
DATA_PATH = '../train.csv'

# Load data using PySpark
df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'false') \
    .schema(schema) \
    .csv(DATA_PATH)

print(f'Dataset successfully loaded!')
print(f'Path: {DATA_PATH}')

Dataset berhasil dimuat!
Path: ../train.csv


## 1.4 View Data Structure

In [ ]:
# Display schema
print('DATASET SCHEMA:')
df.printSchema()

SCHEMA DATASET:
root
 |-- key: string (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- pickup_longitude: float (nullable = true)
 |-- pickup_latitude: float (nullable = true)
 |-- dropoff_longitude: float (nullable = true)
 |-- dropoff_latitude: float (nullable = true)
 |-- passenger_count: integer (nullable = true)



In [ ]:
# Display first 5 rows
print('FIRST 5 ROWS:')
df.show(5, truncate=False)

5 BARIS PERTAMA:
+-----------------------------+-----------+-----------------------+----------------+---------------+-----------------+----------------+---------------+
|key                          |fare_amount|pickup_datetime        |pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+-----------------------------+-----------+-----------------------+----------------+---------------+-----------------+----------------+---------------+
|2009-06-15 17:26:21.0000001  |4.5        |2009-06-15 17:26:21 UTC|-73.844315      |40.721317      |-73.84161        |40.712276       |1              |
|2010-01-05 16:52:16.0000002  |16.9       |2010-01-05 16:52:16 UTC|-74.016045      |40.711304      |-73.97927        |40.782005       |1              |
|2011-08-18 00:35:00.00000049 |5.7        |2011-08-18 00:35:00 UTC|-73.982735      |40.76127       |-73.99124        |40.75056        |2              |
|2012-04-21 04:30:42.0000001  |7.7        |2012-04-21 04:30:42 UTC|-73.

In [ ]:
# Number of rows and columns
total_rows = df.count()
total_cols = len(df.columns)

print('DATASET INFORMATION:')
print(f'Number of rows  : {total_rows:,}')
print(f'Number of columns  : {total_cols}')
print(f'Column names    : {df.columns}')

INFORMASI DATASET:
Jumlah baris  : 50,000
Jumlah kolom  : 8
Nama kolom    : ['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']


## 1.5 Data Type Conversion

Column `pickup_datetime` needs to be converted from String to Timestamp for time-based analysis.

In [ ]:
# Convert pickup_datetime to Timestamp
df = df.withColumn(
    'pickup_datetime',
    F.to_timestamp(F.col('pickup_datetime'), 'yyyy-MM-dd HH:mm:ss z')
)

print('SCHEMA AFTER CONVERSION:')
df.printSchema()

SCHEMA SETELAH KONVERSI:
root
 |-- key: string (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- pickup_longitude: float (nullable = true)
 |-- pickup_latitude: float (nullable = true)
 |-- dropoff_longitude: float (nullable = true)
 |-- dropoff_latitude: float (nullable = true)
 |-- passenger_count: integer (nullable = true)



In [ ]:
# Verify conversion succeeded
df.select('key', 'pickup_datetime', 'fare_amount').show(5, truncate=False)

+-----------------------------+-------------------+-----------+
|key                          |pickup_datetime    |fare_amount|
+-----------------------------+-------------------+-----------+
|2009-06-15 17:26:21.0000001  |2009-06-16 00:26:21|4.5        |
|2010-01-05 16:52:16.0000002  |2010-01-05 23:52:16|16.9       |
|2011-08-18 00:35:00.00000049 |2011-08-18 07:35:00|5.7        |
|2012-04-21 04:30:42.0000001  |2012-04-21 11:30:42|7.7        |
|2010-03-09 07:51:00.000000135|2010-03-09 14:51:00|5.3        |
+-----------------------------+-------------------+-----------+
only showing top 5 rows


## 1.6 Initial Missing Values Check

In [ ]:
# Count null values per column
print('NULL COUNT PER COLUMN:')
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show()

JUMLAH NULL PER KOLOM:
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|key|fare_amount|pickup_datetime|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|  0|          0|              0|               0|              0|                0|               0|              0|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+



## 1.7 Basic Statistics

In [ ]:
# Descriptive statistics for numeric columns
print('DESCRIPTIVE STATISTICS:')
numeric_cols = ['fare_amount', 'pickup_longitude', 'pickup_latitude',
                'dropoff_longitude', 'dropoff_latitude', 'passenger_count']
df.select(numeric_cols).describe().show()

STATISTIK DESKRIPTIF:
+-------+------------------+------------------+-----------------+------------------+----------------+------------------+
|summary|       fare_amount|  pickup_longitude|  pickup_latitude| dropoff_longitude|dropoff_latitude|   passenger_count|
+-------+------------------+------------------+-----------------+------------------+----------------+------------------+
|  count|             50000|             50000|            50000|             50000|           50000|             50000|
|   mean|11.364171422414966| -72.5097555850712|39.93375897353329|-72.50461601842936|39.9262514114378|           1.66784|
| stddev|  9.68555751823222|10.393860223603454|6.224857181308011|10.407569606011792|6.01473670016731|1.2891947001362873|
|    min|              -5.0|         -75.42385|        -74.00689|         -84.65424|       -74.00638|                 0|
|    max|             200.0|          40.78347|        401.08334|          40.85103|        43.41519|                 6|
+-------+-

## 1.8 Save Data as Parquet

Parquet format is more efficient for subsequent processing (better compression, faster column access).

In [ ]:
# Save data as CSV first (easier alternative)
print("Converting Spark DataFrame to Pandas")
pandas_df = df.toPandas()

# Create data folder in parent directory
import os
parent_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(parent_dir, 'data', 'raw')
os.makedirs(data_dir, exist_ok=True)

# Save as CSV
csv_file = os.path.join(data_dir, 'nyc_taxi_data.csv')
pandas_df.to_csv(csv_file, index=False)

print(f'Data successfully saved as CSV to: {csv_file}')
print(f'Total rows: {len(pandas_df)}')
print(f'Total columns: {len(pandas_df.columns)}')

# Display file info
import os.path
file_size = os.path.getsize(csv_file) / (1024*1024)  # Convert to MB
print(f'File size: {file_size:.2f} MB')

Mengkonversi Spark DataFrame ke Pandas
Data berhasil disimpan sebagai CSV ke: d:\Big Data\New York City Taxi Fare Prediction\data\raw\nyc_taxi_data.csv
Total rows: 50000
Total columns: 8
File size: 4.53 MB


In [ ]:
# Verification: Reload from CSV to ensure data is saved correctly
print("Verifying saved data")
df_loaded = pd.read_csv(csv_file)
print(f'Data successfully loaded back from: {csv_file}')
print(f'Total rows: {len(df_loaded)}')
print(f'Total columns: {len(df_loaded.columns)}')
print(f'\nColumns: {list(df_loaded.columns)}')

Verifikasi data yang tersimpan
Data berhasil dimuat kembali dari: d:\Big Data\New York City Taxi Fare Prediction\data\raw\nyc_taxi_data.csv
Total rows: 50000
Total columns: 8

Kolom: ['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']


## Summary

| Item | Description |
|------|-------------|
| Total Rows | 50,000 |
| Total Columns | 8 |
| Target Variable | `fare_amount` |
| Output Format | CSV (`data/raw/`) |

Data is ready for the **Exploratory Data Analysis (EDA)** stage in Notebook 02.

In [ ]:
# Stop SparkSession at the end (optional, if not continuing to another notebook)
# spark.stop()
print('Notebook 01 complete!')

Notebook 01 selesai!
